# core

> Minimal IPython notebook integration for an LLM `%%prompt` cell magic. Inspired by fast.ai's solveit.

The whole module is built around three observations:

1. An `.ipynb` is just JSON. Cells have sources, outputs, and (since nbformat 4.5) stable ids.
2. JupyterLab tells the kernel which cell is executing via `parent_header.metadata.cellId`. That's enough to locate ourselves in the notebook without text matching.
3. Cell outputs persist in the JSON on disk. If a prompt cell already has output, re-running it shouldn't cost tokens.

In [1]:
#| default_exp core

In [2]:
#| hide
from nbdev.showdoc import *

## Imports

In [3]:
#| export
from IPython.display import Markdown, display
from IPython.core.magic import register_cell_magic

from openai import AzureOpenAI

from pathlib import Path
import argparse, json, os, shlex

## System prompt

Kept short and honest about the actual environment. The model takes its tone, structure, and capabilities cues from here, so lies about features we don't have (tool use, variable injection, etc.) leak into responses.

In [4]:
#| export
system_prompt = """You are an AI assistant embedded inside an IPython notebook through a `%%prompt` cell magic. The notebook is a linear sequence of markdown, code, and prompt cells executed in a single persistent kernel — state from code cells carries forward. When the user runs a prompt cell, every cell above it (sources and captured outputs) is sent to you as the conversation history; previous prompt cells appear as your prior assistant turns. The dialog *is* the notebook.

Be concise, direct, and incremental. Match response length to the question. Do not pad, restate, or end with "let me know if...". Use fenced code blocks with language tags. Default to idiomatic Python — comprehensions, broadcasting, fastcore-style brevity. Short single-line docstrings; no inline comments unless a constraint is genuinely non-obvious.

Your knowledge cutoff is January 2026."""

## Locating the current notebook

Two-step strategy. JupyterLab ≥ 3.5 sets `JPY_SESSION_NAME` to the notebook's path; use it to determine the current notebook's path.

In [5]:
os.environ.get("JPY_SESSION_NAME")

'/Users/pablo/src/nbdialog/nbs/00_core.ipynb'

In [6]:
#| export
def _current_notebook(cell_id=None):
    "Resolve the path of the notebook this kernel is serving."
    name = os.environ.get("JPY_SESSION_NAME")
    if name and Path(name).exists(): return Path(name)
    raise RuntimeError("Cannot determine current notebook; save the file or set JPY_SESSION_NAME")

## Cells → chat messages

Prompt cells contribute one `user` message (the prompt body with the `%%prompt` line stripped) and, when they already have rendered output, one `assistant` message replaying that output. Code and markdown cells contribute their source as `user`, plus any captured outputs as a follow-up `user` message. The cell whose id matches `up_to_id` is the boundary — included for its prompt but never for its (stale) cached response, since that response is what the current call is producing.

In [7]:
#| export
def _join(x): return "".join(x) if isinstance(x, list) else x

In [8]:
#| export
def _is_prompt(cell):
    "True if `cell` is a code cell whose source starts with the %%prompt magic."
    if cell.get("cell_type") != "code": return False
    return _join(cell.get("source", "")).lstrip().startswith("%%prompt")

In [9]:
#| export
def _strip_magic(src):
    "Drop the leading %%prompt line from a prompt cell's source."
    return "\n".join(_join(src).splitlines()[1:])

In [10]:
#| export
def _output_text(out):
    "Extract a readable string from one nbformat output entry (for context, not response)."
    if out.get("output_type") == "stream": return _join(out.get("text", []))
    data = out.get("data", {})
    if "text/markdown" in data: return _join(data["text/markdown"])
    if "text/plain"    in data: return _join(data["text/plain"])
    return ""

def _response_markdown(cell):
    "Return the model's rendered markdown response stored in this cell's outputs, or None."
    for o in cell.get("outputs", []):
        data = o.get("data", {})
        if "text/markdown" in data: return _join(data["text/markdown"])
    return None

In [11]:
#| export
def notebook_to_messages(cells, up_to_id, system=None):
    "Build chat messages from `cells` up to and including the cell with id `up_to_id`."
    msgs = [{"role": "system", "content": system or system_prompt}]
    for c in cells:
        src = _join(c["source"])
        is_current = c.get("id") == up_to_id
        if _is_prompt(c):
            msgs.append({"role": "user", "content": _strip_magic(src)})
            if not is_current:
                resp = _response_markdown(c)
                if resp: msgs.append({"role": "assistant", "content": resp})
        else:
            msgs.append({"role": "user", "content": src})
            for o in c.get("outputs", []):
                txt = _output_text(o)
                if txt: msgs.append({"role": "user", "content": f"# Output:\n{txt}"})
        if is_current: break
    return msgs

Sanity check on the demo notebook:

In [12]:
demo = json.loads(Path("small_demo.ipynb").read_text())["cells"]
last = demo[-1]["id"]
[(m["role"], m["content"][:60]) for m in notebook_to_messages(demo, last)]

[('system', 'You are an AI assistant embedded inside an IPython notebook '),
 ('user', '# Small demo\n\nSmall test subject used to understand and test'),
 ('user', 'from nbdialog.core import *'),
 ('user', 'write me hello world in python, like a pirate!'),
 ('assistant', '```python\nprint("Ahoy, world!")\n```'),
 ('user', 'def hello():\n    print("Hello user!")\n\nhello()'),
 ('user', '# Output:\nHello user!\n'),
 ('user', 'which is better?'),
 ('assistant',
  'Yours is better for actual code.\n\nThe pirate version is bett'),
 ('user', '')]

## Azure client

Lazy so that importing `nbdialog.core` doesn't require `AZURE_API_KEY` to be set (handy for offline doc builds and for tests).

In [13]:
#| export
AZURE_OPENAI_DEPLOYMENT = "gpt-5.4"
AZURE_OPENAI_ENDPOINT   = "https://pablo-ml1b1csr-eastus2.cognitiveservices.azure.com"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"

_client = None
def _get_client():
    global _client
    if _client is None:
        _client = AzureOpenAI(
            api_key=os.environ["AZURE_API_KEY"],
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_version=AZURE_OPENAI_API_VERSION,
        )
    return _client

## The `%%prompt` magic

Flow on every execution:

1. Parse the magic line (`-f` / `--force`).
2. Read this cell's id from `parent_header.metadata.cellId`.
3. If the on-disk copy of the cell already has rendered output and `--force` was not passed, replay that output and stop — no API call.
4. Otherwise build the message list from all cells up to and including this one, call the model, and display the response.

The cache lives in the `.ipynb` file itself: outputs are persisted by the normal Jupyter save flow, so they survive kernel restarts and Run-All-from-top, which is the whole point of this feature.

In [14]:
#| export
def _parse_prompt_args(line):
    "Parse the magic line, e.g. `%%prompt -f`."
    p = argparse.ArgumentParser(prog="%%prompt", add_help=False)
    p.add_argument("-f", "--force", action="store_true",
                   help="bypass cache and call the model even if this cell has output")
    return p.parse_args(shlex.split(line or ""))

In [15]:
#| export
@register_cell_magic
def prompt(line, cell):
    "Send the notebook-so-far to the LLM and render its reply as markdown."
    args = _parse_prompt_args(line)
    ip = get_ipython()
    cell_id = ip.parent_header.get("metadata", {}).get("cellId")
    if not cell_id:
        raise RuntimeError("No cellId in parent_header — requires JupyterLab ≥ 3.5")

    nb_path = _current_notebook(cell_id)
    cells = json.loads(nb_path.read_text())["cells"]
    current = next((c for c in cells if c.get("id") == cell_id), None)

    if not args.force and current:
        cached = _response_markdown(current)
        if cached:
            display(Markdown(cached))
            return

    messages = notebook_to_messages(cells, up_to_id=cell_id)
    resp = _get_client().chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        max_completion_tokens=16384,
    )
    display(Markdown(resp.choices[0].message.content))

## Build

In [16]:
#| hide
import nbdev; nbdev.nbdev_export()